# 05 - Content-Based Filtering

Train a content-based nearest-neighbor recommender and evaluate recommendation quality.

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, save_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.utils.sparsefuncs import inplace_csr_column_scale

warnings.filterwarnings("ignore")

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "content_based"
REPORTS_DIR = PROJECT_DIR / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## Load Engineered Features

In [2]:
tracks = pd.read_csv(PROCESSED_DIR / "content_tracks_engineered.csv")
feature_matrix = load_npz(MODEL_DIR / "engineered_feature_matrix.npz")

with open(MODEL_DIR / "engineered_feature_names.json", "r", encoding="utf-8") as file:
    feature_names = json.load(file)

print(f"Tracks: {tracks.shape}")
print(f"Feature matrix: {feature_matrix.shape}")
print(f"Feature names: {len(feature_names):,}")

Tracks: (89740, 40)
Feature matrix: (89740, 4163)
Feature names: 4,163


## Apply TF-IDF Feature Weights

In [3]:
NUMERIC_WEIGHT = 1.0
CATEGORICAL_WEIGHT = 1.0
TEXT_TFIDF_WEIGHT = 2.0

feature_groups = pd.Series(feature_names).str.extract(r"^(numeric|categorical|text)__", expand=False)
feature_weights = np.select(
    [
        feature_groups.eq("numeric"),
        feature_groups.eq("categorical"),
        feature_groups.eq("text"),
    ],
    [NUMERIC_WEIGHT, CATEGORICAL_WEIGHT, TEXT_TFIDF_WEIGHT],
    default=1.0,
).astype(float)

weighted_feature_matrix = feature_matrix.tocsr(copy=True)
inplace_csr_column_scale(weighted_feature_matrix, feature_weights)
weighted_feature_matrix = normalize(weighted_feature_matrix, norm="l2", copy=False)

weight_config = {
    "numeric_weight": NUMERIC_WEIGHT,
    "categorical_weight": CATEGORICAL_WEIGHT,
    "text_tfidf_weight": TEXT_TFIDF_WEIGHT,
}

print(f"Weighted matrix: {weighted_feature_matrix.shape}")
print(weight_config)

Weighted matrix: (89740, 4163)
{'numeric_weight': 1.0, 'categorical_weight': 1.0, 'text_tfidf_weight': 2.0}


## Train Content-Based Model

In [4]:
model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=51, n_jobs=-1)
model.fit(weighted_feature_matrix)

track_id_to_index = dict(zip(tracks["track_id"], tracks["content_index"]))
index_to_track_id = dict(zip(tracks["content_index"].astype(str), tracks["track_id"]))

print("TF-IDF weighted KNN content model trained.")

TF-IDF weighted KNN content model trained.


## Recommend Similar Tracks

In [5]:
def recommend_by_track_id(track_id: str, top_n: int = 10) -> pd.DataFrame:
    if track_id not in track_id_to_index:
        raise ValueError(f"Unknown track_id: {track_id}")

    track_index = track_id_to_index[track_id]
    distances, indices = model.kneighbors(weighted_feature_matrix[track_index], n_neighbors=top_n + 1)

    rec_indices = indices.ravel()[1:]
    rec_distances = distances.ravel()[1:]

    columns = ["track_id", "track_name", "artists", "album_name", "track_genre", "popularity"]
    recommendations = tracks.iloc[rec_indices][columns].copy()
    recommendations["similarity_score"] = 1 - rec_distances
    return recommendations.reset_index(drop=True)

sample_track = tracks.iloc[0]
print(f"Input: {sample_track['track_name']} - {sample_track['artists']}")
recommend_by_track_id(sample_track["track_id"], top_n=10)

Input: Comedy - Gen Hoshino


,track_id,track_name,artists,album_name,track_genre,popularity,similarity_score
0,6x15KrIZn8qZMAByvhPAZU,Los 90,Natos y Waor,Luna llena,spanish,57,0.739118
1,7LiGptffUbVAtbxzQnvBi9,OH NO!,Cody Lovaas;ROZES,Pull Out Couch,chill,57,0.736725
2,186AzR054q9nSWYSI3qr8D,Losing You,Christian Kuria,Borderline,chill,59,0.732376
3,4Nw7kywWurWS6ceinn1cHK,Baby Powder,Jenevieve,Division,chill,68,0.716352
4,7jIAttgQTpLDoNtykIQXjH,Blister In The Sun,Violent Femmes,Violent Femmes,acoustic,71,0.714812
5,6dZCuf6SGn2rh9q94JBLlv,Idhu Varai,Yuvan Shankar Raja;Ajesh;Andrea Jeremiah,Goa (Original Motion Picture Soundtrack),k-pop,64,0.710805
6,4MIKclDDgZgzuaATP5yOjW,On the Low,Justin Park,Places Like Home,chill,63,0.708441
7,391ld8zBXxJRJlspCe6kUR,LA DON’T LOOK GOOD ON U,ASTN,LA DON’T LOOK GOOD ON U,chill,57,0.708410
8,7JqpYvigO9X4QY3W9QTArZ,Arabu Naadu,Haricharan;Yuvan Shankar Raja,Thottal Poo Malarum,pop-film,61,0.705671
9,2AilvPt1AQ8PixlCKqkjfj,Elevated,Shubh,Elevated,hip-hop,76,0.705309


## Evaluate Model

In [6]:
def evaluate_content_model(sample_size: int = 1000, k: int = 10, random_state: int = 42) -> dict:
    sample = tracks.sample(min(sample_size, len(tracks)), random_state=random_state)

    genre_hits = []
    artist_hits = []
    avg_similarities = []
    unique_recommended = set()

    for _, row in sample.iterrows():
        source_index = track_id_to_index[row["track_id"]]
        distances, indices = model.kneighbors(weighted_feature_matrix[source_index], n_neighbors=k + 1)
        rec_indices = indices.ravel()[1:]
        rec_distances = distances.ravel()[1:]
        recs = tracks.iloc[rec_indices]

        genre_hits.append((recs["track_genre"].values == row["track_genre"]).mean())
        artist_hits.append((recs["artists"].values == row["artists"]).mean())
        avg_similarities.append((1 - rec_distances).mean())
        unique_recommended.update(rec_indices.tolist())

    return {
        "sample_size": int(len(sample)),
        "k": int(k),
        "genre_precision_at_k": float(np.mean(genre_hits)),
        "artist_precision_at_k": float(np.mean(artist_hits)),
        "average_similarity_at_k": float(np.mean(avg_similarities)),
        "catalog_coverage": float(len(unique_recommended) / len(tracks)),
    }

metrics = evaluate_content_model(sample_size=1000, k=10, random_state=42)
pd.Series(metrics)

sample_size                1000.000000
k                            10.000000
genre_precision_at_k          0.706100
artist_precision_at_k         0.160900
average_similarity_at_k       0.832579
catalog_coverage              0.104056
dtype: float64

## Baseline Check

In [7]:
popular_tracks = tracks.sort_values("popularity", ascending=False).head(10)

def popularity_baseline_genre_score(sample_size: int = 1000, random_state: int = 42) -> float:
    sample = tracks.sample(min(sample_size, len(tracks)), random_state=random_state)
    scores = []
    for _, row in sample.iterrows():
        baseline_recs = popular_tracks[popular_tracks["track_id"] != row["track_id"]].head(10)
        scores.append((baseline_recs["track_genre"].values == row["track_genre"]).mean())
    return float(np.mean(scores))

baseline_genre_precision = popularity_baseline_genre_score(sample_size=1000, random_state=42)
print(f"Popularity baseline genre precision@10: {baseline_genre_precision:.4f}")
print(f"Content model genre precision@10:       {metrics['genre_precision_at_k']:.4f}")

Popularity baseline genre precision@10: 0.0093
Content model genre precision@10:       0.7061


## Save Model and Metrics

In [8]:
model_path = MODEL_DIR / "content_based_knn_tfidf_weighted_model.joblib"
weighted_matrix_path = MODEL_DIR / "tfidf_weighted_feature_matrix.npz"
weight_config_path = MODEL_DIR / "tfidf_weight_config.json"
mapping_path = MODEL_DIR / "content_model_mappings.json"
metrics_path = REPORTS_DIR / "content_based_metrics.json"

joblib.dump(model, model_path)
save_npz(weighted_matrix_path, weighted_feature_matrix)

with open(mapping_path, "w", encoding="utf-8") as file:
    json.dump({"track_id_to_index": track_id_to_index, "index_to_track_id": index_to_track_id}, file, indent=2)

with open(weight_config_path, "w", encoding="utf-8") as file:
    json.dump(weight_config, file, indent=2)

metrics_with_baseline = dict(metrics)
metrics_with_baseline["popularity_baseline_genre_precision_at_10"] = baseline_genre_precision

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics_with_baseline, file, indent=2)

print("Saved content-based model and metrics.")
print(model_path)
print(weighted_matrix_path)
print(weight_config_path)
print(metrics_path)

Saved content-based model and metrics.
d:\ml\sportify-recommendation\models\content_based\content_based_knn_tfidf_weighted_model.joblib
d:\ml\sportify-recommendation\models\content_based\tfidf_weighted_feature_matrix.npz
d:\ml\sportify-recommendation\models\content_based\tfidf_weight_config.json
d:\ml\sportify-recommendation\reports\content_based_metrics.json


## Summary

In [9]:
summary = pd.DataFrame({
    "metric": list(metrics_with_baseline.keys()),
    "value": list(metrics_with_baseline.values()),
})
summary

,metric,value
0,sample_size,1000.000000
1,k,10.000000
2,genre_precision_at_k,0.706100
3,artist_precision_at_k,0.160900
4,average_similarity_at_k,0.832579
5,catalog_coverage,0.104056
6,popularity_baseline_genre_precision_at_10,0.009300
